# *Regressão Logística* (NumPy)

In [1]:
import numpy as np
import pandas as pd
import sys
import pickle
sys.path.append('.')

from text_features import clean_text, TFIDFVectorizer, encode_labels, labels_to_onehot, decode_labels, CLASS_NAMES
from logistic_regression import LogisticRegression
from metrics import accuracy, f1_macro

In [2]:
TRAIN_PATH = '../datasets/dataset_train.csv'
TEST_PATH  = '../datasets/dataset_test.csv'
VALIDATION_PATH = '../datasets/dataset_val.csv'

df_train = pd.read_csv(TRAIN_PATH,sep=';')
df_test  = pd.read_csv(TEST_PATH, sep=';')
df_val = pd.read_csv(VALIDATION_PATH,sep=';')

print(f'treino: {len(df_train)} exemplos')
print(f'teste:  {len(df_test)}  exemplos')
print()
print('distribuição de treino:')
print(df_train['Label'].value_counts())

treino: 17500 exemplos
teste:  3750  exemplos

distribuição de treino:
Label
Anthropic    3500
Meta         3500
Human        3500
Openai       3500
Google       3500
Name: count, dtype: int64


## *pré-processamento*

In [3]:
# ── limpeza de texto ───────────────────────────────────────────────────────
train_texts  = [clean_text(t) for t in df_train['Text'].tolist()]
train_labels = df_train['Label'].str.lower().tolist()

test_texts   = [clean_text(t) for t in df_test['Text'].tolist()]
test_labels  = df_test['Label'].str.lower().tolist()

# ── TF-IDF ─────────────────────────────────────────────────────────────────
MAX_WORDS = 10000
vec = TFIDFVectorizer(max_words=MAX_WORDS)
X_train = vec.fit_transform(train_texts)
X_test  = vec.transform(test_texts)

# ── labels para one-hot encoding ───────────────────────────────────────────────────
Y_train = labels_to_onehot(encode_labels(train_labels))
Y_test  = labels_to_onehot(encode_labels(test_labels))

print(f'X_train: {X_train.shape}  Y_train: {Y_train.shape}')
print(f'X_test:  {X_test.shape}   Y_test:  {Y_test.shape}')

X_train: (17500, 10000)  Y_train: (17500, 5)
X_test:  (3750, 10000)   Y_test:  (3750, 5)


## *treino do modelo*

In [ ]:
X_train_dense = X_train.toarray() if not isinstance(X_train, np.ndarray) else X_train

model = LogisticRegression(
    X=X_train_dense,
    y=Y_train,
    n_classes=5,
    standardize=False,
    l2_lambda=1e-4
)

model.build_model(alpha=0.1, iters=300, batch_size=32)

[Epoch 20/300] cost: 0.6646
[Epoch 40/300] cost: 0.5065
[Epoch 60/300] cost: 0.4328


## *evaluation*

In [ ]:
X_train_dense = X_train.toarray() if not isinstance(X_train, np.ndarray) else X_train
X_test_dense  = X_test.toarray() if not isinstance(X_test, np.ndarray) else X_test

train_acc, train_f1 = model.score(X_train_dense, Y_train)
test_acc, test_f1  = model.score(X_test_dense,  Y_test)

print(f'[TRAIN] accuracy: {train_acc:.4f}  f1-macro: {train_f1:.4f}')
print(f'[TEST] accuracy: {test_acc:.4f}   f1-macro: {test_f1:.4f}')

In [ ]:
model.save('../models/model_logreg.npz')

with open('../vectorizers/vectorizer_logreg.pkl', 'wb') as f:
    pickle.dump(vec, f)

print('modelo e vectorizer guardados.')

## *validation*

In [ ]:
with open('../vectorizers/vectorizer_logreg.pkl', 'rb') as f:
    vec = pickle.load(f)

model = LogisticRegression(n_classes=5)
model.load('../models/model_logreg.npz')

val_texts  = [clean_text(t) for t in df_val['Text'].tolist()]
val_labels = df_val['Label'].str.lower().tolist()

X_val = vec.transform(val_texts)
X_val_dense = X_val.toarray() if not isinstance(X_val, np.ndarray) else X_val
Y_val = labels_to_onehot(encode_labels(val_labels))

val_acc, val_f1 = model.score(X_val_dense, Y_val)
print(f'accuracy: {val_acc:.4f}  f1-macro: {val_f1:.4f}')

preds_idx = [model.predict(x) for x in X_val_dense]
preds = decode_labels(preds_idx)

df_results = pd.DataFrame({'ID': df_val['ID'], 'Predicted': preds, 'True': df_val['Label']})
print(df_results.to_string(index=False))